# 01_silver_testing_and_validation

## Purpose

Validate the Silver foundation before building Gold outputs.

This notebook checks whether the cleaned Silver tables are safe enough to use for business logic and reporting.

## Expected checks

- Silver table existence
- row counts
- key null checks
- invalid value checks
- join-readiness checks
- validation summary output

## Main idea

Gold outputs should only be built after the Silver layer has been tested and confirmed as trustworthy.

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

catalog = "vattenfall_dev"
schema = "refined"

silver_tables = [
    f"{catalog}.{schema}.silver_market_prices",
    f"{catalog}.{schema}.silver_weather",
    f"{catalog}.{schema}.silver_grid_events",
    f"{catalog}.{schema}.silver_asset_reference",
    f"{catalog}.{schema}.silver_regional_operations_base",
]

In [0]:
validation_rows = []

for table_name in silver_tables:
    df = spark.table(table_name)

    validation_rows.append(
        Row(
            table_name=table_name,
            row_count=df.count(),
            column_count=len(df.columns),
        )
    )

silver_validation_df = spark.createDataFrame(validation_rows)

display(silver_validation_df)

In [0]:
empty_tables = silver_validation_df.filter(F.col("row_count") <= 0)

if empty_tables.count() > 0:
    display(empty_tables)
    raise ValueError("Silver validation failed: one or more Silver tables are empty.")

print("Silver table existence and row count validation passed.")

In [0]:
null_checks = {
    f"{catalog}.{schema}.silver_market_prices": [
        "region", "market_type", "price_eur_mwh", "volume_mwh", "report_day"
    ],
    f"{catalog}.{schema}.silver_weather": [
        "region", "temperature_c", "wind_speed_kmh", "precipitation_mm", "weather_alert_level", "report_day"
    ],
    f"{catalog}.{schema}.silver_grid_events": [
        "event_id", "region", "asset_id", "event_type", "severity", "severity_band", "duration_minutes", "event_day"
    ],
    f"{catalog}.{schema}.silver_asset_reference": [
        "asset_id", "asset_name", "region", "asset_type"
    ],
    f"{catalog}.{schema}.silver_regional_operations_base": [
        "event_id", "asset_id", "region", "event_day"
    ],
}

null_check_rows = []

for table_name, columns in null_checks.items():
    df = spark.table(table_name)

    for column_name in columns:
        null_check_rows.append(
            Row(
                table_name=table_name,
                column_name=column_name,
                null_count=df.filter(F.col(column_name).isNull()).count()
            )
        )

null_check_df = spark.createDataFrame(null_check_rows)

display(null_check_df)

In [0]:
market_df = spark.table(f"{catalog}.{schema}.silver_market_prices")
weather_df = spark.table(f"{catalog}.{schema}.silver_weather")
grid_df = spark.table(f"{catalog}.{schema}.silver_grid_events")
asset_df = spark.table(f"{catalog}.{schema}.silver_asset_reference")

invalid_check_rows = [
    Row(
        check_name="market_price_or_volume_missing",
        invalid_count=market_df.filter(
            F.col("price_eur_mwh").isNull() | F.col("volume_mwh").isNull()
        ).count()
    ),
    Row(
        check_name="negative_wind_speed",
        invalid_count=weather_df.filter(F.col("wind_speed_kmh") < 0).count()
    ),
    Row(
        check_name="negative_grid_event_duration",
        invalid_count=grid_df.filter(F.col("duration_minutes") < 0).count()
    ),
    Row(
        check_name="duplicate_asset_id",
        invalid_count=(
            asset_df
            .groupBy("asset_id")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
    ),
]

invalid_check_df = spark.createDataFrame(invalid_check_rows)

display(invalid_check_df)

In [0]:
grid_df = spark.table(f"{catalog}.{schema}.silver_grid_events")
asset_df = spark.table(f"{catalog}.{schema}.silver_asset_reference")

grid_asset_join_df = (
    grid_df.alias("g")
    .join(asset_df.alias("a"), on="asset_id", how="left")
)

missing_asset_matches = grid_asset_join_df.filter(F.col("a.asset_name").isNull()).count()

print("Grid rows:", grid_df.count())
print("Rows without asset reference match:", missing_asset_matches)

In [0]:
print("Silver foundation validation completed.")
print("The Silver layer is ready for Day 4 Gold business logic.")